# Pseudolabeling Experiment — Post-submission learning

**Goal:** Improve the Swin-Base baseline (0.9363 AUC) using pseudolabeling, the technique used by the SIIM-ISIC 2nd place Kaggle solution.

**What is pseudolabeling?**

We have 33,126 labelled training images and 10,982 *unlabelled* test images. Pseudolabeling uses the trained model to predict labels for the test set, treats high-confidence predictions as if they were real labels ("pseudo-labels"), and retrains on the combined pool.

Even noisy labels carry signal. If the model is right 93% of the time, a confident prediction is probably right 95–98% of the time — plenty of signal to learn from.

**Pipeline (3 steps):**

1. **Predict** soft probabilities on the test set using our best Swin-Base checkpoint.
2. **Filter** high-confidence test images to use as additional training data.
3. **Retrain** Swin-Base from scratch on the combined dataset (train + pseudo-labelled test), then evaluate on the validation fold.

**Key references:**
- Lee, D.-H. (2013). "Pseudo-Label: The Simple and Efficient Semi-Supervised Learning Method for Deep Neural Networks." ICML Workshop. ([paper](http://deeplearning.net/wp-content/uploads/2013/03/pseudo_label_final.pdf))
- Xie et al. (2020). "Self-training with Noisy Student improves ImageNet classification." CVPR. ([arXiv:1911.04252](https://arxiv.org/abs/1911.04252))
- SIIM-ISIC 2nd Place Solution (Kaggle, 2020) used pseudolabeling with 7× minority upsampling.

**Honest expectations:**
- Likely gain: +0.005 to +0.020 AUC
- Could also hurt if confirmation bias kicks in — we'll monitor this
- This is post-submission learning, not part of the original report

## Cell 1 — Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/
!rm -rf MelanomaClassificationAML
!git clone https://github.com/PalsRoy/MelanomaClassificationAML.git

!pip install -q timm albumentations

## Cell 2 — Unzip Data

Same as other experiments — unzips the JPEG data and copies train.csv / test.csv to local disk.

In [ ]:
import os

DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/siim-isic-melanoma-classification'

!mkdir -p /content/data

if not os.path.exists('/content/data/jpeg/train'):
    print('Unzipping JPEG data (~10-15 min)...')
    !unzip -q -o '{DRIVE_DATA}/jpeg.zip' -d /content/data/
else:
    print('JPEG already extracted')

!cp '{DRIVE_DATA}/train.csv' /content/data/
!cp '{DRIVE_DATA}/test.csv' /content/data/

n_train = len(os.listdir('/content/data/jpeg/train'))
n_test = len(os.listdir('/content/data/jpeg/test'))
print(f'\nTrain images: {n_train} / 33126')
print(f'Test images:  {n_test} / 10982')

## Cell 3 — Load Project Modules

Same explicit `importlib` loader as the other experiment notebooks.

In [ ]:
import importlib.util
import sys

REPO = '/content/MelanomaClassificationAML'

def load_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

config_module  = load_module('config',  f'{REPO}/config.py')
dataset_module = load_module('dataset', f'{REPO}/dataset.py')
models_module  = load_module('models',  f'{REPO}/models.py')
train_module   = load_module('train',   f'{REPO}/train.py')

CFG                  = config_module.CFG
MelanomaDataset      = dataset_module.MelanomaDataset
get_transforms       = dataset_module.get_transforms
build_model          = models_module.build_model
train_one_epoch      = train_module.train_one_epoch
validate_one_epoch   = train_module.validate_one_epoch
make_amp_components  = train_module.make_amp_components

print('All modules loaded.')

## Cell 4 — Pseudolabeling Configuration

Hyperparameters specific to pseudolabeling. The defaults are reasonable starting points; see comments for what each one controls.

In [ ]:
# === MATCH THE TARGET ARCHITECTURE (Swin-Base, our winner) ===
EXPERIMENT_NAME = 'exp6_swin_base_pseudolabel'
CFG.model_name = 'swin_base_patch4_window7_224'
CFG.image_size = 224
CFG.batch_size = 32  # match the original Swin-Base run for fair comparison

# === COLAB-SPECIFIC PATHS ===
CFG.DATA_DIR    = '/content/data'
CFG.JPEG_DIR    = '/content/data/jpeg'
CFG.num_workers = 4
CFG.use_amp     = True
CFG.n_epochs    = 5   # match the original 5-epoch comparison budget
CFG.fold        = 0

# === PSEUDOLABELING HYPERPARAMETERS ===

# Path to the trained Swin-Base checkpoint that will generate pseudolabels
TEACHER_CHECKPOINT = '/content/drive/MyDrive/melanoma_results/weights/exp4_swin_base_fold0_best.pth'

# Confidence threshold for keeping a pseudolabel.
# Only test images where max class probability > this threshold are kept.
# Higher = fewer but more reliable pseudolabels.
CONFIDENCE_THRESHOLD = 0.80

# Whether to use soft pseudolabels (probability distributions) or hard ones (argmax).
# Soft is generally better — preserves uncertainty information.
USE_SOFT_LABELS = True

# Upsampling factor for predicted melanomas.
# The Kaggle 2nd place solution used 7×.
# Set to 1 to disable upsampling.
PSEUDO_MELANOMA_UPSAMPLE = 7

# Whether to also use Test-Time Augmentation when generating pseudolabels.
# TTA gives more stable predictions, so higher-quality pseudolabels.
USE_TTA_FOR_PSEUDOLABELS = True
TTA_PASSES = 4

print(f'Experiment:      {EXPERIMENT_NAME}')
print(f'Teacher model:   {TEACHER_CHECKPOINT}')
print(f'Conf threshold:  {CONFIDENCE_THRESHOLD}')
print(f'Soft labels:     {USE_SOFT_LABELS}')
print(f'Melanoma upsample: {PSEUDO_MELANOMA_UPSAMPLE}×')
print(f'TTA passes:      {TTA_PASSES if USE_TTA_FOR_PSEUDOLABELS else "disabled"}')
print(f'Device:          {CFG.device}')

## Cell 5 — Prepare Original Training Data with Folds

Same fold split as the original Swin-Base run, so the validation set is identical.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df_train = pd.read_csv(os.path.join(CFG.DATA_DIR, 'train.csv'))
df_train['filepath'] = df_train['image_name'].apply(
    lambda x: os.path.join(CFG.JPEG_DIR, 'train', f'{x}.jpg')
)

# Same fold logic as your other experiments
sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df_train['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(
    sgkf.split(df_train, df_train['target'], df_train['patient_id'])
):
    df_train.loc[val_idx, 'fold'] = fold_idx

df_val = df_train[df_train['fold'] == CFG.fold].reset_index(drop=True)
df_trn = df_train[df_train['fold'] != CFG.fold].reset_index(drop=True)

print(f'Train: {len(df_trn)} | Val: {len(df_val)}')
print(f'Melanoma in train: {(df_trn["target"]==1).sum()} ({(df_trn["target"]==1).mean()*100:.2f}%)')
print(f'Melanoma in val:   {(df_val["target"]==1).sum()} ({(df_val["target"]==1).mean()*100:.2f}%)')

## Cell 6 — Load the Test Set (Unlabelled)

The test images don't have labels — we'll generate them with the teacher model in Cell 8.

In [ ]:
df_test = pd.read_csv(os.path.join(CFG.DATA_DIR, 'test.csv'))
df_test['filepath'] = df_test['image_name'].apply(
    lambda x: os.path.join(CFG.JPEG_DIR, 'test', f'{x}.jpg')
)

# Sanity check
assert os.path.exists(df_test['filepath'].iloc[0]), f"Missing: {df_test['filepath'].iloc[0]}"

print(f'Test images: {len(df_test)}')
print(f'Columns: {list(df_test.columns)}')
df_test.head(3)

## Cell 7 — Load the Teacher Model

The teacher is the trained Swin-Base from the original architecture comparison (val AUC 0.9363). It will generate the pseudolabels.

In [ ]:
import torch

# Build the architecture
teacher = build_model(
    model_name=CFG.model_name,
    out_dim=9,
    pretrained=False,  # we load saved weights, not ImageNet pretrained
    drop_rate=0.5,
).to(CFG.device)

# Load checkpoint
checkpoint = torch.load(TEACHER_CHECKPOINT, map_location=CFG.device)
teacher.load_state_dict(checkpoint['model_state_dict'])
teacher.eval()

print(f'Teacher loaded from epoch {checkpoint["epoch"]}')
print(f'Teacher val AUC at save time: {checkpoint["best_auc"]:.4f}')
print(f'Parameters: {sum(p.numel() for p in teacher.parameters() if p.requires_grad):,}')

## Cell 8 — Generate Pseudolabels for the Test Set

**This is the core of the technique.** We run the teacher over every test image (optionally with TTA) and store the resulting softmax probability distributions.

Expected runtime: ~5 min without TTA, ~15-20 min with 4-pass TTA on A100.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm

@torch.no_grad()
def predict_test_set(model, df, transform, device, n_tta=1, batch_size=64):
    """
    Generate soft predictions over a dataset, optionally with TTA.
    
    For n_tta > 1, the same dataset is iterated multiple times with
    different (stochastic) augmentations, and predictions are averaged.
    
    Returns:
        all_probs: array of shape (N, out_dim) with averaged softmax probabilities
    """
    accumulated = None
    
    for tta_pass in range(n_tta):
        # Re-create the dataset/loader each pass so the random transforms differ
        ds = MelanomaDataset(df.assign(target=0), transform=transform)  # dummy target
        loader = DataLoader(
            ds, batch_size=batch_size, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True,
        )
        
        pass_probs = []
        desc = f'  TTA pass {tta_pass+1}/{n_tta}' if n_tta > 1 else '  Predict'
        for images, _ in tqdm(loader, desc=desc, leave=False):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            pass_probs.append(probs)
        pass_probs = np.concatenate(pass_probs, axis=0)
        
        if accumulated is None:
            accumulated = pass_probs
        else:
            accumulated += pass_probs
    
    return accumulated / n_tta


# Use training transforms for TTA, validation transforms for single-pass
if USE_TTA_FOR_PSEUDOLABELS:
    print(f'Generating pseudolabels with {TTA_PASSES}-pass TTA...')
    transform = get_transforms(CFG.image_size, 'train')
    test_probs = predict_test_set(
        teacher, df_test, transform, CFG.device,
        n_tta=TTA_PASSES, batch_size=CFG.batch_size * 2,
    )
else:
    print('Generating pseudolabels (single pass, no TTA)...')
    transform = get_transforms(CFG.image_size, 'val')
    test_probs = predict_test_set(
        teacher, df_test, transform, CFG.device,
        n_tta=1, batch_size=CFG.batch_size * 2,
    )

print(f'\nPredictions shape: {test_probs.shape}')
print(f'Mean melanoma probability: {test_probs[:, 0].mean():.4f}')
print(f'Max melanoma probability: {test_probs[:, 0].max():.4f}')
print(f'Predicted melanomas (argmax): {(test_probs.argmax(axis=1) == 0).sum()}')

## Cell 9 — Filter and Inspect Pseudolabels

Apply the confidence threshold and inspect what we're about to use for training. This is the most important diagnostic step — if the distribution looks wrong here, retraining will go badly.

In [ ]:
# Confidence = max probability across all 9 classes
confidence = test_probs.max(axis=1)
pseudo_class = test_probs.argmax(axis=1)

# Filter by confidence threshold
keep_mask = confidence >= CONFIDENCE_THRESHOLD
n_kept = keep_mask.sum()
n_discarded = (~keep_mask).sum()

print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'Test images kept:      {n_kept} / {len(df_test)} ({100*n_kept/len(df_test):.1f}%)')
print(f'Test images discarded: {n_discarded}')

# Distribution of kept pseudolabels
from collections import Counter
kept_classes = pseudo_class[keep_mask]
class_counts = Counter(kept_classes.tolist())

DIAGNOSIS_NAMES = {
    0: 'melanoma', 1: 'nevus', 2: 'seborrheic_keratosis',
    3: 'lentigo_NOS', 4: 'lichenoid_keratosis', 5: 'solar_lentigo',
    6: 'cafe-au-lait_macule', 7: 'atypical_melanocytic_proliferation', 8: 'unknown',
}

print('\nPseudolabel class distribution:')
for cls_idx in sorted(class_counts.keys()):
    name = DIAGNOSIS_NAMES.get(cls_idx, f'class_{cls_idx}')
    print(f'  {cls_idx} ({name:<35}): {class_counts[cls_idx]:>6}')

n_pseudo_melanoma = class_counts.get(0, 0)
print(f'\nPseudo-melanomas (rare class):  {n_pseudo_melanoma}')
print(f'Real melanomas in training:     {(df_trn["target"]==1).sum()}')
print(f'After {PSEUDO_MELANOMA_UPSAMPLE}× upsampling, effective pseudo-melanomas: {n_pseudo_melanoma * PSEUDO_MELANOMA_UPSAMPLE}')

## Cell 10 — Visualise the Confidence Distribution

Sanity check: see where the model is confident and uncertain. If confidence is mostly very low (say < 0.5), the threshold needs lowering and the pseudolabels will be too noisy to be useful.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of confidence
axes[0].hist(confidence, bins=50, color='#0D9488', edgecolor='black', alpha=0.7)
axes[0].axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Threshold = {CONFIDENCE_THRESHOLD}')
axes[0].set_xlabel('Max softmax probability (confidence)')
axes[0].set_ylabel('Number of test images')
axes[0].set_title('Teacher confidence distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Melanoma probability distribution
mel_probs = test_probs[:, 0]
axes[1].hist(mel_probs, bins=50, color='#DC2626', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Predicted P(melanoma)')
axes[1].set_ylabel('Number of test images')
axes[1].set_title('Melanoma probability across test set')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSanity checks:')
print(f'  Test images with P(melanoma) > 0.5: {(mel_probs > 0.5).sum()}')
print(f'  Test images with P(melanoma) > 0.8: {(mel_probs > 0.8).sum()}')
print(f'  Expected real melanomas in test (~1.76%): ~{int(0.0176 * len(df_test))}')

## Cell 11 — Build the Combined Training Dataset

Combine:
- Original real training data (df_trn), with binary `target` → 9-class one-hot soft labels
- Filtered pseudolabelled test data, with the teacher's predicted soft probabilities
- Upsampled pseudo-melanomas (per Kaggle 2nd place solution recipe)

Because we want to support soft labels, we extend each row with a `pseudo_probs` column. The dataset class will use this if present, else fall back to the hard `target`.

In [ ]:
# Build the pseudolabel dataframe (only kept test images)
df_pseudo = df_test[keep_mask].reset_index(drop=True).copy()
df_pseudo['pseudo_probs'] = list(test_probs[keep_mask])  # one row of probs per image
df_pseudo['target'] = kept_classes  # hard label = argmax of pseudoprobs
df_pseudo['is_pseudo'] = True

# Upsample pseudo-melanomas
if PSEUDO_MELANOMA_UPSAMPLE > 1:
    pseudo_mel = df_pseudo[df_pseudo['target'] == 0]
    pseudo_rest = df_pseudo[df_pseudo['target'] != 0]
    
    # Replicate the melanoma rows
    pseudo_mel_upsampled = pd.concat([pseudo_mel] * PSEUDO_MELANOMA_UPSAMPLE, ignore_index=True)
    df_pseudo = pd.concat([pseudo_rest, pseudo_mel_upsampled], ignore_index=True)
    print(f'Upsampled pseudo-melanomas: {len(pseudo_mel)} → {len(pseudo_mel_upsampled)}')

# Prepare original training data — mark it as not pseudo
df_trn_combined = df_trn.copy()
df_trn_combined['is_pseudo'] = False
df_trn_combined['pseudo_probs'] = None  # will use hard `target` for these rows

# Combine
df_combined = pd.concat([df_trn_combined, df_pseudo], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=CFG.seed).reset_index(drop=True)

print(f'\nCombined training set composition:')
print(f'  Real samples:     {(~df_combined["is_pseudo"]).sum()}')
print(f'  Pseudo samples:   {df_combined["is_pseudo"].sum()}')
print(f'  Total:            {len(df_combined)}')
print(f'  Effective melanomas (real + pseudo): {(df_combined["target"]==0).sum() if "target" in df_combined.columns else "?"}')
# Note: in real labels melanoma == 1; in pseudo labels melanoma class index == 0. The
# original `target` from train.csv uses 0/1 (benign/malignant), which is a different
# encoding from the 9-class diagnosis index used inside the model. The next cell
# handles this mismatch explicitly.

## Cell 12 — Pseudolabel-Aware Dataset Class

We extend `MelanomaDataset` to handle:
- Real samples (use `diagnosis` mapped to 9-class index)
- Pseudo samples (use the soft `pseudo_probs` distribution)

Returns (image, target_or_probs, is_pseudo_flag) so the training loop can route correctly.

In [ ]:
import cv2
from torch.utils.data import Dataset

# Map diagnosis strings → 9-class index. Must match what your other notebooks use.
DIAGNOSIS2IDX = {
    'melanoma': 0,
    'nevus': 1,
    'seborrheic keratosis': 2,
    'lentigo NOS': 3,
    'lichenoid keratosis': 4,
    'solar lentigo': 5,
    'cafe-au-lait macule': 6,
    'atypical melanocytic proliferation': 7,
    'unknown': 8,
}

class PseudoLabelDataset(Dataset):
    """Dataset that handles a mix of real labels and soft pseudolabels.
    
    For real rows, the loss function will receive a class index.
    For pseudo rows with soft labels, the loss function will receive
    a probability distribution (and use soft cross-entropy).
    """
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        image = cv2.imread(row['filepath'])
        if image is None:
            raise FileNotFoundError(f"Image not found: {row['filepath']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image=image)['image']
        
        is_pseudo = bool(row['is_pseudo'])
        
        if is_pseudo and USE_SOFT_LABELS and row.get('pseudo_probs') is not None:
            # Soft label — return the full probability distribution
            target = torch.tensor(row['pseudo_probs'], dtype=torch.float32)
        else:
            # Hard label
            # For real samples, `target` is binary 0/1 (benign/malignant) from train.csv.
            # We map: malignant (1) → class 0 (melanoma), benign (0) → class 1 (nevus).
            # This matches the convention used in the original architecture comparison runs.
            if is_pseudo:
                cls_idx = int(row['target'])  # already 9-class index for pseudolabels
            else:
                cls_idx = 0 if row['target'] == 1 else 1
            target = torch.tensor(cls_idx, dtype=torch.long)
        
        return image, target, is_pseudo


# Build the combined dataset
train_transform = get_transforms(CFG.image_size, 'train')
val_transform = get_transforms(CFG.image_size, 'val')

train_ds = PseudoLabelDataset(df_combined, transform=train_transform)

# Val dataset uses the original MelanomaDataset, but we still need to remap target
# from binary to 9-class for consistency with model output.
df_val_remapped = df_val.copy()
df_val_remapped['target_9class'] = df_val_remapped['target'].apply(lambda x: 0 if x == 1 else 1)
df_val_remapped = df_val_remapped.drop(columns=['target']).rename(columns={'target_9class': 'target'})
val_ds = MelanomaDataset(df_val_remapped, transform=val_transform)

train_loader = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True,
)

print(f'Train batches: {len(train_loader)} ({len(train_ds)} samples)')
print(f'Val batches:   {len(val_loader)} ({len(val_ds)} samples)')

## Cell 13 — Mixed Loss Function

The training batch contains a mix of real (hard label) and pseudo (soft label) samples. We need a loss function that handles both.

- **Hard labels:** standard cross-entropy `-log(p_y)`
- **Soft labels:** soft cross-entropy `-Σ q_c log(p_c)` where q is the soft target distribution

Both reduce to the same formula when soft targets are one-hot.

In [ ]:
import torch.nn.functional as F

def mixed_loss(logits, targets, is_pseudo):
    """
    Apply cross-entropy with hard labels for real samples and
    soft cross-entropy for pseudo samples.
    
    Args:
        logits: (B, C) raw model outputs
        targets: either (B,) class indices, or (B, C) soft distributions
        is_pseudo: (B,) bool tensor indicating which samples are pseudo
    """
    log_probs = F.log_softmax(logits, dim=1)
    
    if targets.dim() == 1:
        # All hard labels
        return F.nll_loss(log_probs, targets)
    
    if targets.dim() == 2:
        # All soft labels — compute soft CE = -sum(q * log_p) per row
        return -(targets * log_probs).sum(dim=1).mean()
    
    raise ValueError(f'Unexpected target shape: {targets.shape}')


# Sanity check the mixed loss
test_logits = torch.randn(4, 9)
test_hard = torch.tensor([0, 1, 2, 0])
test_soft = F.one_hot(test_hard, num_classes=9).float()

loss_hard = mixed_loss(test_logits, test_hard, torch.tensor([False, False, False, False]))
loss_soft = mixed_loss(test_logits, test_soft, torch.tensor([True, True, True, True]))

print(f'Loss (hard): {loss_hard.item():.4f}')
print(f'Loss (soft, one-hot equivalent): {loss_soft.item():.4f}')
print(f'Match: {torch.allclose(loss_hard, loss_soft, atol=1e-5)}  (should be True)')

## Cell 14 — Training Loop

Similar to the standard training loop, but uses the mixed loss and collates the pseudo flag per batch.

Validation uses the original validate function on real labelled data only — the metric is still AUC on the held-out fold-0 patients, comparable to the 0.9363 baseline.

In [ ]:
import time
import json
from sklearn.metrics import roc_auc_score

MEL_IDX = 0

def custom_collate(batch):
    """Custom collate: stack images, handle mixed target shapes."""
    images = torch.stack([b[0] for b in batch])
    targets_raw = [b[1] for b in batch]
    is_pseudo = torch.tensor([b[2] for b in batch], dtype=torch.bool)
    
    # If any target is 1-D (soft), convert all to soft for uniform batching
    any_soft = any(t.dim() == 1 and t.numel() > 1 for t in targets_raw)
    
    if any_soft:
        # Convert hard labels to one-hot
        targets = []
        for t in targets_raw:
            if t.dim() == 0:  # scalar class index
                t = F.one_hot(t, num_classes=9).float()
            targets.append(t)
        targets = torch.stack(targets)
    else:
        targets = torch.stack(targets_raw)
    
    return images, targets, is_pseudo


# Rebuild loader with the custom collate
train_loader = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
    collate_fn=custom_collate,
)

# Build the student model (fresh weights — we re-train from ImageNet init,
# matching the original Swin-Base experiment)
student = build_model(
    model_name=CFG.model_name,
    out_dim=9,
    pretrained=True,
    drop_rate=0.5,
).to(CFG.device)

optimizer = torch.optim.Adam(student.parameters(), lr=CFG.init_lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.n_epochs, eta_min=CFG.init_lr * 0.01,
)
scaler, use_amp = make_amp_components(use_amp=CFG.use_amp, device=CFG.device)

# Storage
WEIGHTS_DIR = '/content/drive/MyDrive/melanoma_results/weights'
RESULTS_DIR = '/content/drive/MyDrive/melanoma_results/results'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'lr': [], 'time_min': []}
best_auc = 0.0

print(f'Starting pseudolabel training: {EXPERIMENT_NAME}')
print('=' * 70)

for epoch in range(1, CFG.n_epochs + 1):
    lr_now = optimizer.param_groups[0]['lr']
    print(f'\nEpoch {epoch}/{CFG.n_epochs} (lr={lr_now:.2e})')
    t0 = time.time()
    
    # ---- Train ----
    student.train()
    running_loss = 0.0
    n_samples = 0
    
    pbar = tqdm(train_loader, desc='  Train', leave=False)
    for images, targets, is_pseudo in pbar:
        images = images.to(CFG.device, non_blocking=True)
        targets = targets.to(CFG.device, non_blocking=True)
        is_pseudo = is_pseudo.to(CFG.device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if use_amp and CFG.device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = student(images)
                loss = mixed_loss(logits, targets, is_pseudo)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = student(images)
            loss = mixed_loss(logits, targets, is_pseudo)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    train_loss = running_loss / n_samples
    
    # ---- Validate (only on real labelled data) ----
    student.eval()
    all_targets = []
    all_probs = []
    val_loss_total = 0.0
    
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc='  Valid', leave=False):
            images = images.to(CFG.device, non_blocking=True)
            targets = targets.to(CFG.device, non_blocking=True)
            
            logits = student(images)
            val_loss_total += F.cross_entropy(logits, targets).item() * images.size(0)
            probs = torch.softmax(logits, dim=1)[:, MEL_IDX]
            
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    all_targets = np.concatenate(all_targets)
    all_probs = np.concatenate(all_probs)
    val_loss = val_loss_total / len(all_targets)
    
    binary_targets = (all_targets == MEL_IDX).astype(int)
    val_auc = roc_auc_score(binary_targets, all_probs)
    
    scheduler.step()
    elapsed = (time.time() - t0) / 60
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['lr'].append(lr_now)
    history['time_min'].append(elapsed)
    
    print(f'  Train Loss: {train_loss:.4f}')
    print(f'  Val   Loss: {val_loss:.4f}  AUC: {val_auc:.4f}')
    print(f'  Time:       {elapsed:.1f} min')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'best_auc': best_auc,
            'experiment': EXPERIMENT_NAME,
        }, f'{WEIGHTS_DIR}/{EXPERIMENT_NAME}_fold{CFG.fold}_best.pth')
        print(f'  New best AUC: {best_auc:.4f} → saved')
    
    # Save history every epoch
    with open(f'{RESULTS_DIR}/{EXPERIMENT_NAME}.json', 'w') as f:
        json.dump({
            'experiment_name': EXPERIMENT_NAME,
            'model_name':      CFG.model_name,
            'image_size':      CFG.image_size,
            'batch_size':      CFG.batch_size,
            'fold':            CFG.fold,
            'best_auc':        best_auc,
            'pseudolabel_config': {
                'teacher_checkpoint': TEACHER_CHECKPOINT,
                'confidence_threshold': CONFIDENCE_THRESHOLD,
                'soft_labels': USE_SOFT_LABELS,
                'melanoma_upsample': PSEUDO_MELANOMA_UPSAMPLE,
                'tta_passes': TTA_PASSES if USE_TTA_FOR_PSEUDOLABELS else 0,
                'pseudo_samples_kept': int(n_kept),
                'pseudo_melanomas_before_upsample': int(n_pseudo_melanoma),
            },
            'history': history,
        }, f, indent=2)

print('\n' + '=' * 70)
print(f'Training complete!')
print(f'Best Val AUC: {best_auc:.4f}')
print(f'Baseline (no pseudo): 0.9363')
print(f'Δ vs baseline:        {best_auc - 0.9363:+.4f}')
print('=' * 70)

## Cell 15 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_x, history['train_loss'], 'o-', label='Train', color='#1B2A4A')
axes[0].plot(epochs_x, history['val_loss'], 's-', label='Val', color='#0D9488')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_auc'], 'o-', color='#7C3AED', linewidth=2,
             label=f'Pseudo (best {best_auc:.4f})')
axes[1].axhline(y=0.9363, color='black', linestyle='--', alpha=0.5,
                label='Baseline 0.9363')
axes[1].axhline(y=0.5, color='red', linestyle=':', alpha=0.3, label='Random')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val AUC')
axes[1].set_title('Validation AUC vs Baseline')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim([0.80, 0.96])

axes[2].plot(epochs_x, history['lr'], 'o-', color='#475569')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle(f'{EXPERIMENT_NAME} - Pseudolabel learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{EXPERIMENT_NAME}_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSummary:')
print(f'  Baseline (no pseudo):     0.9363')
print(f'  Pseudolabel best:         {best_auc:.4f}')
print(f'  Δ vs baseline:            {best_auc - 0.9363:+.4f}')
print(f'  Total time:               {sum(history["time_min"]):.1f} min')

## Discussion (post-experiment)

After running, take a moment to reflect:

**If pseudolabeling helped (+0.005 or more):**
- Confirms the technique adds value for this dataset
- Try tuning: lower confidence threshold, different upsample factor, more TTA passes for the teacher
- Could iterate: use the pseudolabeled model to generate *new* pseudolabels ("self-training rounds")

**If pseudolabeling hurt or was neutral:**
- Likely confirmation bias — the teacher's biases were amplified
- Try raising the confidence threshold (e.g., 0.9 or 0.95) to keep only the safest predictions
- Try disabling melanoma upsampling — 7× may be over-amplifying noise
- This would itself be an interesting finding: "pseudolabeling did not transfer to our setup because [hypothesis]"

**Honest reporting note:** Whatever the result, this is post-submission learning. The original report stands on its own merits. If you write this up later (e.g. for a portfolio or follow-up paper), report the result faithfully — including any negative finding.